# 42 - Round-based active learning for a deeper "headline query" benchmark

Every query in notebook 40's expansion got roughly the same shallow depth (~10-14 gold-labelled candidates each). That is enough to fix the out-of-distribution problem for the silver-labelling classifier (confirmed in notebook 41), but it is thin for reporting honest per-query Recall@k/NDCG@k as an actual evaluation benchmark, deeper judged pools matter more for that (see the pooling-depth literature already cited, Section~4).

Rather than deepening all 101 queries (unaffordable in review time, not just API cost), this deepens 14 "headline" queries chosen for methodological diversity, not just topic variety:

- **Geography, not yet covered**: query 11 (tech companies in Munich), 12 (manufacturing in Baden-Wurttemberg), 15 (logistics companies in the US)
- **Direct probe of the known BM25 failure mode**: query 34 (venture capital firms)
- **Adjacent-category disambiguation pairs**: query 14/27 (pharmaceutical vs biotech), 91/92 (investment management vs wealth management)
- **Acronym-heavy queries**: query 66 (CRM software), 72 (managed service providers / MSPs)
- **Niche/emerging industries**: query 99 (vertical farming), 101 (alternative protein manufacturers)
- **Additional sector diversity**: query 56 (defense contractors), 82 (public relations agencies)

Unlike notebook 40's one-shot batch, this is genuinely iterative: label a round, retrain, re-rank the *remaining* candidates with the *updated* classifier (uncertainty estimates go stale otherwise), and repeat. A **fixed validation set**, judged once with the strong judge pair and then reused for free, tracks whether each additional round is still earning its cost, so there is an actual, cheap stopping signal instead of an arbitrary round count.

In [1]:
import os, json, time
import numpy as np
import pandas as pd
import requests
from pathlib import Path
from dotenv import load_dotenv
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

OUTPUT_DIR = Path("result/42_headline_query_deepening")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HEADLINE_QUERY_IDS = [11, 12, 15, 34, 14, 27, 66, 72, 99, 101, 56, 82, 91, 92]
RELEVANT_THRESHOLD = 2  # same strict "highly relevant" definition used throughout
ROUND_K = 20  # candidates per headline query added per round -- keep small, disagreements need manual review
VALIDATION_N_PER_QUERY = 8

feature_cols = ["score_minilm", "score_linq", "score_gte", "score_bm25",
                "invrank_minilm", "invrank_linq", "invrank_gte", "invrank_bm25",
                "reranker_score", "n_channels"]

# Base data: current expanded gold standard (all 101 queries, notebook 40) and the full feature table.
base_gold = pd.read_json("result/40_active_learning_labeling_queue/expanded_gold_labels.json")
features = pd.read_csv("result/30_learned_fusion_ranker/scored_candidates.csv")
corpus = pd.read_csv("dataset/company_corpus.csv")

# company_corpus.csv is one row per domain (see notebook 40) -- join company metadata on domain
# alone, and query text on query_id alone, never on the (query_id, domain) pair.
meta_cols = ["domain", "name", "country", "state", "district", "municipality",
             "organization_type", "organization_size", "summary", "summary_keywords", "nace_code"]
company_meta = corpus[meta_cols].drop_duplicates(subset="domain")
query_text = corpus[["query_id", "query"]].drop_duplicates(subset="query_id")


def train_classifier(gold_df):
    labeled = gold_df.merge(features, on=["query_id", "domain"], how="inner")
    labeled["relevant"] = (labeled["gold_label"] >= RELEVANT_THRESHOLD).astype(int)
    X = labeled[feature_cols].values
    y = labeled["relevant"].values
    gbdt = HistGradientBoostingClassifier(max_iter=150, max_depth=4, class_weight="balanced", random_state=0)
    gbdt.fit(X, y)
    return gbdt, labeled


def attach_metadata(df):
    df = df.merge(company_meta, on="domain", how="left")
    df = df.merge(query_text, on="query_id", how="left")
    return df


ENRICHED_JUDGE_PROMPT_TEMPLATE = """You are judging search result relevance for a company search engine.

Search query: "{query}"

Candidate company:
Name: {name}
Country: {country}
State/region: {state}
District: {district}
Municipality: {municipality}
Organization type: {organization_type}
Organization size: {organization_size}
NACE industry code: {nace_code}
Summary: {summary}
Summary keywords: {summary_keywords}

Rate how relevant this company is to the search query, using exactly one of these labels:
2 = highly relevant (a strong, direct match for the query)
1 = partially relevant (related but not a strong direct match)
0 = not relevant

Respond with ONLY a JSON object: {{"label": <0, 1, or 2>, "reason": "<one short sentence>"}}"""


def build_prompt(row):
    fields = {c: row.get(c, "") if pd.notna(row.get(c, "")) else "unknown" for c in
              ["query", "name", "country", "state", "district", "municipality",
               "organization_type", "organization_size", "nace_code", "summary", "summary_keywords"]}
    return ENRICHED_JUDGE_PROMPT_TEMPLATE.format(**fields)


def parse_judge_reply(text):
    try:
        start, end = text.index("{"), text.rindex("}") + 1
        parsed = json.loads(text[start:end])
        return int(parsed["label"]), parsed.get("reason", "")
    except (ValueError, KeyError, json.JSONDecodeError):
        return None, f"UNPARSEABLE: {text[:200]}"


def judge_openai_cheap(prompt):
    resp = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}", "Content-Type": "application/json"},
        json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0},
        timeout=60,
    )
    resp.raise_for_status()
    return parse_judge_reply(resp.json()["choices"][0]["message"]["content"])


def judge_claude_cheap(prompt):
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={"x-api-key": ANTHROPIC_API_KEY, "anthropic-version": "2023-06-01", "Content-Type": "application/json"},
        json={"model": "claude-haiku-4-5-20251001", "max_tokens": 200, "messages": [{"role": "user", "content": prompt}]},
        timeout=60,
    )
    resp.raise_for_status()
    return parse_judge_reply(resp.json()["content"][0]["text"])


def judge_openai_strong(prompt):
    resp = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}", "Content-Type": "application/json"},
        json={"model": "gpt-5.4", "messages": [{"role": "user", "content": prompt}], "temperature": 0},
        timeout=60,
    )
    resp.raise_for_status()
    return parse_judge_reply(resp.json()["choices"][0]["message"]["content"])


def judge_claude_strong(prompt):
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={"x-api-key": ANTHROPIC_API_KEY, "anthropic-version": "2023-06-01", "Content-Type": "application/json"},
        json={"model": "claude-sonnet-5", "max_tokens": 1024, "messages": [{"role": "user", "content": prompt}]},
        timeout=60,
    )
    resp.raise_for_status()
    # Sonnet 5 runs adaptive thinking by default, so content[0] may be a "thinking" block,
    # not "text" -- find the actual text block instead of assuming it's first.
    content_blocks = resp.json()["content"]
    text_block = next((b["text"] for b in content_blocks if b.get("type") == "text"), None)
    if text_block is None:
        return None, f"NO TEXT BLOCK: {content_blocks}"
    return parse_judge_reply(text_block)


CHEAP_JUDGES = {"openai": judge_openai_cheap, "claude": judge_claude_cheap}
print(f"Headline queries: {HEADLINE_QUERY_IDS} ({len(HEADLINE_QUERY_IDS)} total)")
print(f"Base gold standard: {len(base_gold)} candidates across {base_gold['query_id'].nunique()} queries")

Headline queries: [11, 12, 15, 34, 14, 27, 66, 72, 99, 101, 56, 82, 91, 92] (14 total)
Base gold standard: 1429 candidates across 101 queries


## Step 1 (one-time): build the fixed validation set

Run this once. Samples a handful of candidates per headline query, judges them with the strong pair, and keeps unanimous agreements as a small held-out benchmark. These candidates are excluded from every later round's candidate pool, so this stays honestly held-out.

In [2]:
validation_path = OUTPUT_DIR / "headline_validation_set.json"

if validation_path.exists():
    validation_set = pd.read_json(validation_path)
    print(f"Validation set already built: {len(validation_set)} candidates. Delete the file to rebuild.")
else:
    gold_keys = set(zip(base_gold["query_id"], base_gold["domain"]))
    headline_pool = features[features["query_id"].isin(HEADLINE_QUERY_IDS)].copy()
    headline_pool = headline_pool[~headline_pool.apply(lambda r: (r["query_id"], r["domain"]) in gold_keys, axis=1)]

    val_sample = (
        headline_pool.groupby("query_id", group_keys=False)
        .apply(lambda g: g.sample(n=min(VALIDATION_N_PER_QUERY, len(g)), random_state=7))
        .reset_index(drop=True)
    )
    val_sample = attach_metadata(val_sample)
    print(f"Validation candidates sampled: {len(val_sample)} across {val_sample['query_id'].nunique()} queries")

    val_cache_path = OUTPUT_DIR / "validation_judge_cache.json"
    val_cache = json.load(open(val_cache_path)) if val_cache_path.exists() else {}

    for i, row in val_sample.iterrows():
        key = f'{row["query_id"]}::{row["domain"]}'
        entry = val_cache.get(key, {})
        prompt = build_prompt(row)
        if "gpt54" not in entry:
            try:
                label, reason = judge_openai_strong(prompt)
                entry["gpt54"] = {"label": label, "reason": reason}
            except (requests.exceptions.RequestException, KeyError, IndexError) as e:
                print(f"  [gpt-5.4] error on {row['domain']}: {e}")
        if "sonnet5" not in entry:
            try:
                label, reason = judge_claude_strong(prompt)
                entry["sonnet5"] = {"label": label, "reason": reason}
            except (requests.exceptions.RequestException, KeyError, IndexError) as e:
                print(f"  [sonnet5] error on {row['domain']}: {e}")
        val_cache[key] = entry
        json.dump(val_cache, open(val_cache_path, "w"), indent=2, default=str)
        if (i + 1) % 25 == 0 or (i + 1) == len(val_sample):
            print(f"  {i+1}/{len(val_sample)} validation candidates judged")
        time.sleep(0.2)

    rows = []
    for i, row in val_sample.iterrows():
        key = f'{row["query_id"]}::{row["domain"]}'
        entry = val_cache.get(key, {})
        gpt_label, sonnet_label = entry.get("gpt54", {}).get("label"), entry.get("sonnet5", {}).get("label")
        if gpt_label is None or sonnet_label is None or gpt_label != sonnet_label:
            continue  # unanimous only, same principle as every spot-check in this thesis
        rows.append({
            "query_id": row["query_id"], "domain": row["domain"],
            "true_relevant": int(gpt_label >= RELEVANT_THRESHOLD),
        })
    validation_set = pd.DataFrame(rows)
    validation_set.to_json(validation_path, orient="records", indent=2)
    print(f"Fixed validation set: {len(validation_set)}/{len(val_sample)} unanimous -> {validation_path}")

/scratch/ipykernel_111257/795882308.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(VALIDATION_N_PER_QUERY, len(g)), random_state=7))


Validation candidates sampled: 112 across 14 queries
  25/112 validation candidates judged
  50/112 validation candidates judged
  75/112 validation candidates judged
  100/112 validation candidates judged
  112/112 validation candidates judged
Fixed validation set: 100/112 unanimous -> result/42_headline_query_deepening/headline_validation_set.json


## Step 2 (repeatable): run one active learning round

Rerun this cell (then the merge cell, then the evaluate cell below) for each new round. Each run automatically detects the current round number from `round_gold_labels.json` and retrains on everything labelled so far, including prior rounds, before ranking what is left by uncertainty.

In [10]:
round_gold_path = OUTPUT_DIR / "round_gold_labels.json"
round_gold_so_far = pd.read_json(round_gold_path) if round_gold_path.exists() else pd.DataFrame(columns=["query_id", "domain", "gold_label", "round"])
CURRENT_ROUND = int(round_gold_so_far["round"].max()) + 1 if len(round_gold_so_far) else 1
print(f"Starting round {CURRENT_ROUND}. Candidates added in prior rounds: {len(round_gold_so_far)}")

# Retrain on everything labelled so far: the base 101-query gold standard plus every completed round.
current_gold = pd.concat([base_gold[["query_id", "domain", "gold_label"]], round_gold_so_far[["query_id", "domain", "gold_label"]]], ignore_index=True)
classifier, _ = train_classifier(current_gold)

# Exclude anything already gold-labelled (base + prior rounds) and the reserved validation set.
excluded_keys = set(zip(current_gold["query_id"], current_gold["domain"])) | set(zip(validation_set["query_id"], validation_set["domain"]))
headline_pool = features[features["query_id"].isin(HEADLINE_QUERY_IDS)].copy()
headline_pool = headline_pool[~headline_pool.apply(lambda r: (r["query_id"], r["domain"]) in excluded_keys, axis=1)]

headline_pool["prob_relevant"] = classifier.predict_proba(headline_pool[feature_cols].values)[:, 1]
headline_pool["uncertainty"] = (headline_pool["prob_relevant"] - 0.5).abs()

round_queue = (
    headline_pool.sort_values("uncertainty")
    .groupby("query_id", group_keys=False)
    .head(ROUND_K)
    .copy()
)
round_queue = attach_metadata(round_queue)
print(f"Round {CURRENT_ROUND} queue: {len(round_queue)} candidates across {round_queue['query_id'].nunique()} queries")

# Judge with the cheap ensemble, same caching pattern as every prior notebook.
judge_cache_path = OUTPUT_DIR / "judge_cache.json"
judge_cache = json.load(open(judge_cache_path)) if judge_cache_path.exists() else {}

for i, row in round_queue.iterrows():
    key = f'{row["query_id"]}::{row["domain"]}'
    entry = judge_cache.get(key, {})
    prompt = build_prompt(row)
    changed = False
    for judge_name, judge_fn in CHEAP_JUDGES.items():
        if judge_name in entry:
            continue
        try:
            label, reason = judge_fn(prompt)
            entry[judge_name] = {"label": label, "reason": reason}
            changed = True
        except (requests.exceptions.RequestException, KeyError, IndexError) as e:
            print(f"  [{judge_name}] error on {row['domain']}: {e}")
    if changed:
        judge_cache[key] = entry
        json.dump(judge_cache, open(judge_cache_path, "w"), indent=2, default=str)
    if (i + 1) % 25 == 0 or (i + 1) == len(round_queue):
        print(f"  {i+1}/{len(round_queue)} judged")
    time.sleep(0.2)

# Split agree/disagree for this round.
rows = []
for _, row in round_queue.iterrows():
    key = f'{row["query_id"]}::{row["domain"]}'
    entry = judge_cache.get(key, {})
    labels = {j: entry[j]["label"] for j in CHEAP_JUDGES if j in entry and entry[j]["label"] is not None}
    if len(labels) < 2:
        continue
    unanimous = len(set(labels.values())) == 1
    rows.append({
        "query_id": row["query_id"], "domain": row["domain"], **{f"{j}_label": v for j, v in labels.items()},
        "agree": unanimous, "gold_label": list(labels.values())[0] if unanimous else None,
    })
round_results = pd.DataFrame(rows)
round_agreed = round_results[round_results["agree"]]
round_disagreed = round_results[~round_results["agree"]]
round_agreed.to_json(OUTPUT_DIR / f"gold_labels_agreed_round{CURRENT_ROUND}.json", orient="records", indent=2)
print(f"Round {CURRENT_ROUND}: {len(round_agreed)} agreed, {len(round_disagreed)} disagreed")

# Build this round's review queue (preserves any existing labels, same fix as notebook 40).
review_path = OUTPUT_DIR / f"review_queue_round{CURRENT_ROUND}.csv"
existing_labels = {}
if review_path.exists():
    existing = pd.read_csv(review_path)
    for _, r in existing.iterrows():
        val = r.get("your_label", "")
        if pd.notna(val) and str(val).strip() != "":
            existing_labels[(int(r["query_id"]), r["domain"])] = val

queue_lookup = round_queue.set_index(["query_id", "domain"])
review_rows = []
for _, r in round_disagreed.iterrows():
    context = queue_lookup.loc[(r["query_id"], r["domain"])]
    row = {
        "query_id": r["query_id"], "query": context["query"], "domain": r["domain"],
        "name": context["name"], "country": context["country"], "state": context["state"],
        "nace_code": context["nace_code"], "summary": context["summary"],
        "openai_label": r.get("openai_label"), "claude_label": r.get("claude_label"),
    }
    row["your_label"] = existing_labels.get((r["query_id"], r["domain"]), "")
    review_rows.append(row)
pd.DataFrame(review_rows).to_csv(review_path, index=False)
print(f"Round {CURRENT_ROUND} review queue -> {review_path} ({len(review_rows)} rows)")
print("Fill in your_label (0, 1, 2, or SKIP), then run the merge cell below.")

Starting round 3. Candidates added in prior rounds: 560
Round 3 queue: 280 candidates across 14 queries
  25/280 judged
  50/280 judged
  75/280 judged
  100/280 judged
  125/280 judged
  150/280 judged
  175/280 judged
  200/280 judged
  225/280 judged
  250/280 judged
  275/280 judged
  280/280 judged
Round 3: 223 agreed, 57 disagreed
Round 3 review queue -> result/42_headline_query_deepening/review_queue_round3.csv (57 rows)
Fill in your_label (0, 1, 2, or SKIP), then run the merge cell below.


## Step 3: merge this round's label

Run only after `review_queue_round{N}.csv` (N shown by the cell above) has every `your_label` filled in.

In [11]:
review_final = pd.read_csv(OUTPUT_DIR / f"review_queue_round{CURRENT_ROUND}.csv")
assert (review_final["your_label"].astype(str).str.strip() != "").all(), \
    f"Fill in every your_label cell in review_queue_round{CURRENT_ROUND}.csv before running this cell."

resolved = review_final[review_final["your_label"].astype(str).str.upper() != "SKIP"].copy()
resolved["gold_label"] = resolved["your_label"].astype(int)

this_round_new = pd.concat([
    round_agreed[["query_id", "domain", "gold_label"]],
    resolved[["query_id", "domain", "gold_label"]],
], ignore_index=True)
this_round_new["round"] = CURRENT_ROUND

round_gold_path = OUTPUT_DIR / "round_gold_labels.json"
round_gold_so_far = pd.read_json(round_gold_path) if round_gold_path.exists() else pd.DataFrame(columns=["query_id", "domain", "gold_label", "round"])
round_gold_updated = pd.concat([round_gold_so_far, this_round_new], ignore_index=True)
round_gold_updated.to_json(round_gold_path, orient="records", indent=2)

print(f"Round {CURRENT_ROUND} added {len(this_round_new)} new gold labels.")
print(f"Total across all rounds so far: {len(round_gold_updated)}")
print(f"Now run the evaluate cell below to check whether this round is still improving accuracy.")

Round 3 added 280 new gold labels.
Total across all rounds so far: 840
Now run the evaluate cell below to check whether this round is still improving accuracy.


## Step 4: evaluate progress against the fixed validation set

Retrains on everything labelled so far and checks accuracy against the Step 1 validation set (no new API cost). Compares to the previous round so you can decide whether to run another round (Step 2 again) or stop.

In [12]:
round_gold_so_far = pd.read_json(OUTPUT_DIR / "round_gold_labels.json")
current_gold = pd.concat([base_gold[["query_id", "domain", "gold_label"]], round_gold_so_far[["query_id", "domain", "gold_label"]]], ignore_index=True)
classifier, _ = train_classifier(current_gold)

val_features = validation_set.merge(features, on=["query_id", "domain"], how="inner")
val_features["pred_relevant"] = (classifier.predict_proba(val_features[feature_cols].values)[:, 1] >= 0.5).astype(int)
accuracy = (val_features["pred_relevant"] == val_features["true_relevant"]).mean()

history_path = OUTPUT_DIR / "round_history.json"
history = json.load(open(history_path)) if history_path.exists() else []
completed_round = int(round_gold_so_far["round"].max()) if len(round_gold_so_far) else 0
history.append({"round": completed_round, "total_added": len(round_gold_so_far), "validation_accuracy": accuracy})
json.dump(history, open(history_path, "w"), indent=2)

print(f"After round {completed_round}: validation accuracy = {accuracy:.1%} (n={len(val_features)})")
if len(history) >= 2:
    prev_accuracy = history[-2]["validation_accuracy"]
    delta = accuracy - prev_accuracy
    print(f"Change since round {history[-2]['round']}: {delta:+.1%}")
    if delta < 0.03:
        print("Gain is small (<3 points) -- consider stopping here rather than running another round.")
    else:
        print("Still improving meaningfully -- another round (Step 2) is likely worth it.")
else:
    print("First round complete -- run Step 2 again for a second round to see whether it's still improving.")

After round 3: validation accuracy = 83.0% (n=100)
Change since round 2: +0.0%
Gain is small (<3 points) -- consider stopping here rather than running another round.


## Gold-labelled candidates per headline query

In [13]:
round_gold_so_far = pd.read_json(OUTPUT_DIR / "round_gold_labels.json") if (OUTPUT_DIR / "round_gold_labels.json").exists() else pd.DataFrame(columns=["query_id", "domain", "gold_label", "round"])
headline_base = base_gold[base_gold["query_id"].isin(HEADLINE_QUERY_IDS)]
all_headline_gold = pd.concat([headline_base[["query_id", "domain"]], round_gold_so_far[["query_id", "domain"]]], ignore_index=True)

counts = all_headline_gold.groupby("query_id").size().reindex(HEADLINE_QUERY_IDS, fill_value=0)
counts_df = counts.reset_index()
counts_df.columns = ["query_id", "gold_labelled_companies"]
counts_df = counts_df.merge(query_text, on="query_id", how="left")
counts_df = counts_df[["query_id", "query", "gold_labelled_companies"]].sort_values("gold_labelled_companies", ascending=False)

print(counts_df.to_string(index=False))
print()
print(f"Total: {counts_df['gold_labelled_companies'].sum()} across {len(counts_df)} headline queries")
print(f"Average: {counts_df['gold_labelled_companies'].mean():.1f} per query (min {counts_df['gold_labelled_companies'].min()}, max {counts_df['gold_labelled_companies'].max()})")

 query_id                                           query  gold_labelled_companies
       11                        tech companies in Munich                       70
       12              manufacturing in Baden-Württemberg                       70
       15                   logistics companies in the US                       70
       34                           venture capital firms                       70
       14                        pharmaceutical companies                       70
       27                               biotech companies                       70
       66 customer relationship management (CRM) software                       70
       72                managed service providers (MSPs)                       70
       99                      vertical farming companies                       70
      101               alternative protein manufacturers                       70
       56                             defense contractors                       70
    